# Lab 04: Embeddings & Vector Search

## Overview
In this lab you will generate text embeddings with Amazon Titan Embeddings V2, create an OpenSearch Serverless vector index, index documents with different chunking strategies, and perform similarity search with metadata filtering and hybrid search. These are the foundational building blocks for Retrieval-Augmented Generation (RAG) pipelines.

## Learning Objectives
- Generate text embeddings with Amazon Titan Embeddings V2 via Bedrock
- Understand vector dimensions, normalization, and cosine similarity
- Implement fixed-size and recursive character chunking strategies
- Create and query an OpenSearch Serverless vector index using knn_vector fields
- Perform metadata filtering and hybrid search (keyword + vector)

## Exam Domain
**Building Generative AI Applications (30%)** — This lab covers Task 2.1 (RAG solutions) focusing on the data preparation stage: chunking documents, generating embeddings, and storing/querying vectors in a search index.

## Architecture Diagram
![Lab 04 Architecture](../assets/diagrams/lab-04-embeddings-vector-search.png)

In [ ]:
%pip install boto3 opensearch-py requests-aws4auth --quiet

In [ ]:
import boto3, json, os, time
import numpy as np

REGION = "us-east-1"

# Environment detection
if os.environ.get("AWS_DEFAULT_REGION") and os.path.exists("/opt/ml"):
    session = boto3.Session(region_name=REGION)
    print("Running in SageMaker Studio")
else:
    session = boto3.Session(region_name=REGION)
    print("Running locally")

bedrock_runtime = session.client("bedrock-runtime")
s3 = session.client("s3")
sts = session.client("sts")
aoss = session.client("opensearchserverless")

ACCOUNT_ID = sts.get_caller_identity()["Account"]
BUCKET = f"aws-genai-lab-{ACCOUNT_ID}"

# Get OpenSearch collection endpoint
collections = aoss.batch_get_collection(names=["genai-lab-vectors"])
COLLECTION_ENDPOINT = collections["collectionDetails"][0]["collectionEndpoint"]
# Remove https:// prefix for opensearch-py host
COLLECTION_HOST = COLLECTION_ENDPOINT.replace("https://", "")

print(f"Account:            {ACCOUNT_ID}")
print(f"S3 bucket:          {BUCKET}")
print(f"OpenSearch endpoint: {COLLECTION_ENDPOINT}")

## A. Understanding Embeddings

**Embeddings** are dense vector representations of text that capture semantic meaning. Unlike keyword-based search, which matches exact words, embeddings place semantically similar texts close together in a high-dimensional vector space — even when they use completely different words.

Amazon Titan Embeddings V2 (`amazon.titan-embed-text-v2:0`) supports:
- **Configurable dimensions** — 256, 512, or 1024 (higher = more detail, more storage)
- **Normalization** — unit-length vectors for efficient cosine similarity computation
- **Up to 8,192 input tokens** per request

Key properties of embedding vectors:
- Two texts about the same topic produce vectors that are **close together** (high cosine similarity)
- Two texts about different topics produce vectors that are **far apart** (low cosine similarity)
- The model captures meaning, not just word overlap — "happy" and "joyful" are close; "bank" (financial) and "bank" (river) are far apart in the right context

### Generating Embeddings with Titan V2

We define a reusable helper function that calls the Titan Embeddings V2 model via Bedrock. Then we generate embeddings for three sample texts — two about Bedrock/LLMs (semantically similar) and one about S3 (semantically different).

In [ ]:
def get_embedding(text, dimensions=1024, normalize=True):
    """Generate a text embedding using Amazon Titan Embeddings V2."""
    response = bedrock_runtime.invoke_model(
        modelId="amazon.titan-embed-text-v2:0",
        contentType="application/json",
        accept="application/json",
        body=json.dumps({
            "inputText": text,
            "dimensions": dimensions,
            "normalize": normalize,
        }),
    )
    return json.loads(response["body"].read())["embedding"]


# Generate embeddings for sample texts
texts = [
    "Amazon Bedrock provides foundation models",
    "Bedrock offers access to large language models",
    "Amazon S3 is an object storage service",
]

embeddings = [get_embedding(t) for t in texts]

print(f"Embedding dimension: {len(embeddings[0])}")
print(f"First 5 values:      {embeddings[0][:5]}")
print(f"Vector norm:          {np.linalg.norm(embeddings[0]):.6f}")  # ~1.0 when normalized

### Cosine Similarity

**Cosine similarity** measures the angle between two vectors. A value of 1.0 means the vectors point in the same direction (identical meaning), 0.0 means they are orthogonal (unrelated), and -1.0 means they point in opposite directions.

When vectors are normalized (unit length), cosine similarity simplifies to a dot product — this is why normalization matters for performance at scale.

In [ ]:
def cosine_similarity(a, b):
    """Compute cosine similarity between two vectors."""
    a, b = np.array(a), np.array(b)
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))


# Compare similarity between our three texts
print("Pairwise cosine similarity:")
print(f"  'Bedrock FM'  vs  'Bedrock LLM' (similar topics):    {cosine_similarity(embeddings[0], embeddings[1]):.4f}")
print(f"  'Bedrock FM'  vs  'S3 storage'  (different topics):  {cosine_similarity(embeddings[0], embeddings[2]):.4f}")
print(f"  'Bedrock LLM' vs  'S3 storage'  (different topics):  {cosine_similarity(embeddings[1], embeddings[2]):.4f}")

### Comparing Embedding Dimensions

Titan V2 supports 256, 512, and 1024 dimensions. Higher dimensions capture more nuance but require more storage and compute. Let's compare how dimension choice affects similarity scores.

In [ ]:
# Compare dimensions: 256 vs 512 vs 1024
text_a = "Amazon Bedrock provides foundation models"
text_b = "Bedrock offers access to large language models"

for dim in [256, 512, 1024]:
    emb_a = get_embedding(text_a, dimensions=dim)
    emb_b = get_embedding(text_b, dimensions=dim)
    sim = cosine_similarity(emb_a, emb_b)
    print(f"  Dimensions: {dim:>4}  |  Similarity: {sim:.4f}  |  Storage per vector: {dim * 4:,} bytes")

## B. Chunking Strategies

Before you can embed and index a document, you need to split it into **chunks** — smaller text segments that fit within the embedding model's token limit and capture coherent units of meaning. Chunking strategy directly impacts retrieval quality.

**Why chunking matters:**
- Embedding models have token limits (Titan V2: 8,192 tokens) — documents must be split to fit
- Chunks that are too large dilute the embedding with multiple topics, reducing search precision
- Chunks that are too small lose context, reducing search recall
- Overlap between chunks prevents information loss at boundaries

| Strategy | How It Works | Best For |
|----------|-------------|----------|
| Fixed-size | Split every N characters with M-character overlap | Simple documents, uniform structure |
| Recursive character | Split by paragraphs, then sentences, then words | Documents with natural structure (reports, articles) |
| Semantic | Split at topic boundaries using embeddings | Long documents with multiple topics |
| Document-aware | Split by headings, sections, or page breaks | Structured documents (PDFs, HTML) |

### Chunk Size Guidelines

| Chunk Size | Tokens (~) | Trade-off |
|-----------|-----------|-----------|
| 200-500 chars | 50-125 | High precision, low context — good for FAQ-style retrieval |
| 500-1000 chars | 125-250 | Balanced precision and context — good general-purpose default |
| 1000-2000 chars | 250-500 | More context per chunk — good for detailed technical documents |
| 2000+ chars | 500+ | Risk of topic mixing — only for very focused single-topic sections |

### Loading a Sample Document

We download a sample document from the lab S3 bucket. The whitepapers were uploaded by `setup-resources.py` during initial setup. For this lab we use a text excerpt rather than the full PDF to keep things simple and cost-effective.

In [ ]:
# Sample documents — we use plain text excerpts for this lab.
# In production you would extract text from PDFs, HTML, or other formats.

documents = {
    "well-architected": """The AWS Well-Architected Framework describes key concepts, design principles, and architectural best practices for designing and running workloads in the cloud. By answering a set of foundational questions, you learn how well your architecture aligns with cloud best practices and are provided guidance for making improvements.

The framework is based on six pillars: Operational Excellence, Security, Reliability, Performance Efficiency, Cost Optimization, and Sustainability. Operational Excellence focuses on running and monitoring systems to deliver business value and continually improving processes and procedures. Security focuses on protecting information and systems through risk assessments and mitigation strategies.

Reliability ensures a workload performs its intended function correctly and consistently when expected. Performance Efficiency focuses on using computing resources efficiently to meet system requirements and maintaining that efficiency as demand changes and technologies evolve. Cost Optimization focuses on avoiding unnecessary costs by understanding spending, selecting the right resources, and scaling to meet business needs without overspending.

Sustainability focuses on minimizing the environmental impacts of running cloud workloads. This includes understanding the impact of services, reducing downstream resource consumption, and applying best practices for energy-efficient architectures. Each pillar includes design principles, best practices, and a set of questions to evaluate your architecture.

The Well-Architected Framework provides a consistent approach for evaluating architectures and implementing designs that will scale over time. AWS provides the Well-Architected Tool in the AWS Management Console to help you review your workloads against the framework.""",

    "bedrock-overview": """Amazon Bedrock is a fully managed service that makes foundation models from leading AI companies available through a unified API. With Amazon Bedrock, you can choose from a wide range of foundation models to find the model best suited for your use case. You can experiment with and evaluate models, privately customize them with your data, and build generative AI applications.

Bedrock supports several model providers including Anthropic (Claude), AI21 Labs (Jurassic), Cohere, Meta (Llama), Stability AI, and Amazon (Titan). Each provider offers models with different capabilities, context windows, and pricing. The Converse API provides a consistent interface across all models, simplifying application code.

Amazon Bedrock offers key features for enterprise AI applications. Knowledge Bases for Amazon Bedrock provides a fully managed RAG solution that connects foundation models to your data sources. Agents for Amazon Bedrock enables you to build AI assistants that can break down tasks, call APIs, and access knowledge bases. Guardrails for Amazon Bedrock lets you implement safeguards to filter harmful content and protect sensitive information.

Fine-tuning allows you to customize a foundation model with your own training data to improve performance on specific tasks. Continued pre-training extends the model's knowledge with additional domain-specific data. Model evaluation helps you compare models using automatic metrics or human evaluation to select the best model for your use case.

Bedrock is serverless, so you do not need to manage infrastructure. It integrates with AWS services like S3, Lambda, CloudWatch, and IAM for building secure, scalable generative AI applications. Pricing is based on input and output tokens processed, with on-demand and provisioned throughput options available.""",

    "rag-patterns": """Retrieval-Augmented Generation (RAG) is a technique that enhances foundation model responses by retrieving relevant information from external knowledge sources before generating an answer. Instead of relying solely on the model's training data, RAG grounds responses in your specific documents, databases, or APIs.

A RAG pipeline consists of two main phases: indexing and retrieval. During the indexing phase, documents are split into chunks, converted to vector embeddings, and stored in a vector database. During the retrieval phase, the user's query is embedded using the same model, similar chunks are retrieved via vector search, and the retrieved context is included in the prompt sent to the foundation model.

Chunking strategy is critical for RAG quality. Fixed-size chunking splits documents at regular intervals with overlap. Recursive chunking respects document structure by splitting at paragraph boundaries first, then sentences, then words. Semantic chunking uses embeddings to detect topic boundaries. The optimal chunk size depends on your documents and use case — typically 500 to 1000 characters works well for general-purpose retrieval.

Vector databases store embeddings and support similarity search. Amazon OpenSearch Serverless provides a managed vector search capability with knn_vector field types, HNSW indexing for fast approximate nearest neighbor search, and metadata filtering. Amazon Aurora with pgvector and Amazon Neptune also support vector storage.

Hybrid search combines vector similarity search with keyword-based search to improve retrieval quality. Vector search captures semantic meaning while keyword search catches exact terms that might be missed by embeddings. Metadata filtering narrows results by document source, date, category, or other attributes, improving both relevance and performance.""",
}

# Print document statistics
for name, text in documents.items():
    print(f"  {name:20s}  |  {len(text):,} chars  |  ~{len(text.split()):,} words")
print(f"\n  Total: {sum(len(t) for t in documents.values()):,} characters")

### Strategy 1: Fixed-Size Chunking

The simplest approach: split the document every N characters with an overlap of M characters. Overlap ensures that sentences at chunk boundaries are not cut in half without context in the next chunk.

In [ ]:
def fixed_size_chunk(text, chunk_size=500, overlap=100):
    """Split text into fixed-size chunks with overlap."""
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunk = text[start:end]
        chunks.append(chunk.strip())
        start += chunk_size - overlap
    return [c for c in chunks if c]  # Remove empty chunks


# Apply fixed-size chunking to the Well-Architected document
fixed_chunks = fixed_size_chunk(documents["well-architected"], chunk_size=500, overlap=100)

print(f"Fixed-size chunking: {len(fixed_chunks)} chunks")
print(f"Chunk size: 500 chars, Overlap: 100 chars\n")

for i, chunk in enumerate(fixed_chunks):
    print(f"--- Chunk {i} ({len(chunk)} chars) ---")
    print(chunk[:150] + "..." if len(chunk) > 150 else chunk)
    print()

### Strategy 2: Recursive Character Chunking

A smarter approach that respects document structure. It tries to split by paragraphs first (double newline), then by sentences (period + space), then by words (space). This preserves coherent units of meaning rather than cutting mid-sentence.

In [ ]:
def recursive_chunk(text, chunk_size=500, overlap=100, separators=None):
    """Split text recursively by paragraph, sentence, then word boundaries."""
    if separators is None:
        separators = ["\n\n", ". ", " "]

    # Base case: text fits in one chunk
    if len(text) <= chunk_size:
        return [text.strip()] if text.strip() else []

    # Try each separator in order of preference
    for sep in separators:
        parts = text.split(sep)
        if len(parts) > 1:
            chunks = []
            current = ""
            for part in parts:
                candidate = current + sep + part if current else part
                if len(candidate) <= chunk_size:
                    current = candidate
                else:
                    if current:
                        chunks.append(current.strip())
                    current = part
            if current:
                chunks.append(current.strip())

            # Add overlap by prepending the tail of the previous chunk
            if overlap > 0 and len(chunks) > 1:
                overlapped = [chunks[0]]
                for i in range(1, len(chunks)):
                    prev_tail = chunks[i - 1][-overlap:]
                    overlapped.append(prev_tail + " " + chunks[i])
                chunks = overlapped

            return [c for c in chunks if c]

    # Fallback: split by character
    return fixed_size_chunk(text, chunk_size, overlap)


# Apply recursive chunking to the Well-Architected document
recursive_chunks = recursive_chunk(documents["well-architected"], chunk_size=500, overlap=100)

print(f"Recursive chunking: {len(recursive_chunks)} chunks")
print(f"Chunk size: 500 chars, Overlap: 100 chars\n")

for i, chunk in enumerate(recursive_chunks):
    print(f"--- Chunk {i} ({len(chunk)} chars) ---")
    print(chunk[:150] + "..." if len(chunk) > 150 else chunk)
    print()

### Comparing Chunking Strategies

Let's chunk all three documents with both strategies and compare the results. We will use recursive chunking for the rest of the lab since it produces more coherent chunks.

In [ ]:
# Chunk all documents using recursive strategy
all_chunks = []
for source, text in documents.items():
    chunks = recursive_chunk(text, chunk_size=500, overlap=100)
    for i, chunk in enumerate(chunks):
        all_chunks.append({
            "text": chunk,
            "source": source,
            "chunk_id": i,
        })

print(f"Total chunks across all documents: {len(all_chunks)}\n")

# Breakdown by source
for source in documents:
    count = sum(1 for c in all_chunks if c["source"] == source)
    avg_len = np.mean([len(c["text"]) for c in all_chunks if c["source"] == source])
    print(f"  {source:20s}  |  {count} chunks  |  avg {avg_len:.0f} chars")

## C. Indexing Documents in OpenSearch

Now we connect to the pre-provisioned **OpenSearch Serverless** collection and create a vector index. OpenSearch Serverless is a managed, auto-scaling search service — you do not need to provision or manage clusters.

The `genai-lab-vectors` collection was created by `setup-resources.py` with the necessary encryption, network, and data-access policies. We create a **knn_vector** index that stores:
- `embedding` — the 1024-dimensional vector from Titan V2
- `text` — the original chunk text for display
- `source` — which document the chunk came from (for metadata filtering)
- `chunk_id` — the chunk's position within its source document

### Connect to OpenSearch Serverless

We use SigV4 authentication (`requests-aws4auth`) with the `aoss` service name. This is required for OpenSearch Serverless — standard OpenSearch username/password authentication is not supported.

In [ ]:
from opensearchpy import OpenSearch, RequestsHttpConnection
from requests_aws4auth import AWS4Auth

credentials = session.get_credentials().get_frozen_credentials()
awsauth = AWS4Auth(
    credentials.access_key,
    credentials.secret_key,
    REGION,
    "aoss",
    session_token=credentials.token,
)

oss_client = OpenSearch(
    hosts=[{"host": COLLECTION_HOST, "port": 443}],
    http_auth=awsauth,
    use_ssl=True,
    verify_certs=True,
    connection_class=RequestsHttpConnection,
)

# Verify connection
info = oss_client.info()
print(f"Connected to OpenSearch: {info['version']['distribution']}")

### Create the Vector Index

The index mapping defines a `knn_vector` field with 1024 dimensions (matching Titan V2 output) and uses the **HNSW** (Hierarchical Navigable Small World) algorithm via the FAISS engine for fast approximate nearest neighbor search.

In [ ]:
INDEX_NAME = "lab-vectors"

# Delete the index if it already exists (from a previous run)
if oss_client.indices.exists(index=INDEX_NAME):
    oss_client.indices.delete(index=INDEX_NAME)
    print(f"Deleted existing index: {INDEX_NAME}")

# Create the vector index
index_body = {
    "settings": {
        "index": {
            "knn": True,
        }
    },
    "mappings": {
        "properties": {
            "embedding": {
                "type": "knn_vector",
                "dimension": 1024,
                "method": {
                    "engine": "faiss",
                    "name": "hnsw",
                },
            },
            "text": {"type": "text"},
            "source": {"type": "keyword"},
            "chunk_id": {"type": "integer"},
        }
    },
}

oss_client.indices.create(index=INDEX_NAME, body=index_body)
print(f"Created index: {INDEX_NAME}")

### Generate Embeddings and Index Documents

We embed each chunk and index it into OpenSearch. This is the most time-consuming and costly step — each chunk requires one Titan V2 API call.

In [ ]:
# Generate embeddings and index each chunk
print(f"Indexing {len(all_chunks)} chunks into '{INDEX_NAME}'...\n")

for idx, chunk in enumerate(all_chunks):
    # Generate embedding
    embedding = get_embedding(chunk["text"])

    # Index into OpenSearch
    doc = {
        "embedding": embedding,
        "text": chunk["text"],
        "source": chunk["source"],
        "chunk_id": chunk["chunk_id"],
    }
    oss_client.index(index=INDEX_NAME, body=doc, id=str(idx))
    print(f"  Indexed chunk {idx:>2}: source={chunk['source']:20s}  chunk_id={chunk['chunk_id']}")

# OpenSearch Serverless indexes are eventually consistent — wait briefly
print("\nWaiting for index to become searchable...")
time.sleep(5)

# Verify document count
count = oss_client.count(index=INDEX_NAME)
print(f"\nDocuments in index: {count['count']}")

## D. Similarity Search

Now we query the vector index with a natural language question. The process mirrors how a RAG pipeline works:
1. **Embed the query** using the same model (Titan V2) that was used to embed the documents
2. **Search the index** using k-nearest neighbor (knn) to find the most similar chunks
3. **Return ranked results** ordered by similarity score

The query and document embeddings must come from the **same model** — mixing models produces incompatible vector spaces and meaningless similarity scores.

In [ ]:
# Query 1: Well-Architected Framework question
query = "What are the pillars of the Well-Architected Framework?"
query_embedding = get_embedding(query)

results = oss_client.search(
    index=INDEX_NAME,
    body={
        "size": 3,
        "query": {
            "knn": {
                "embedding": {
                    "vector": query_embedding,
                    "k": 3,
                }
            }
        },
    },
)

print(f"Query: {query}\n")
print(f"Top {len(results['hits']['hits'])} results:")
print("=" * 80)
for hit in results["hits"]["hits"]:
    print(f"Score: {hit['_score']:.4f}  |  Source: {hit['_source']['source']}  |  Chunk: {hit['_source']['chunk_id']}")
    print(f"  {hit['_source']['text'][:200]}...")
    print()

In [ ]:
# Query 2: Bedrock-specific question — should surface bedrock-overview chunks
query_2 = "How does Amazon Bedrock support fine-tuning and customization?"
query_embedding_2 = get_embedding(query_2)

results_2 = oss_client.search(
    index=INDEX_NAME,
    body={
        "size": 3,
        "query": {
            "knn": {
                "embedding": {
                    "vector": query_embedding_2,
                    "k": 3,
                }
            }
        },
    },
)

print(f"Query: {query_2}\n")
print(f"Top {len(results_2['hits']['hits'])} results:")
print("=" * 80)
for hit in results_2["hits"]["hits"]:
    print(f"Score: {hit['_score']:.4f}  |  Source: {hit['_source']['source']}  |  Chunk: {hit['_source']['chunk_id']}")
    print(f"  {hit['_source']['text'][:200]}...")
    print()

## E. Metadata Filtering & Hybrid Search

### Metadata Filtering

In production RAG systems, you rarely want to search across all documents. **Metadata filtering** restricts the search to a subset of documents based on attributes like source, date, category, or access level. This improves both relevance (fewer irrelevant results) and performance (smaller search space).

OpenSearch supports metadata filtering via `bool` queries that combine `knn` with `filter` clauses.

In [ ]:
# Filtered search — only search within the "well-architected" document
query = "What are the pillars of the Well-Architected Framework?"
query_embedding = get_embedding(query)

filtered_results = oss_client.search(
    index=INDEX_NAME,
    body={
        "size": 3,
        "query": {
            "bool": {
                "must": [
                    {"knn": {"embedding": {"vector": query_embedding, "k": 3}}}
                ],
                "filter": [
                    {"term": {"source": "well-architected"}}
                ],
            }
        },
    },
)

print(f"Query: {query}")
print(f"Filter: source = 'well-architected'\n")
print(f"Top {len(filtered_results['hits']['hits'])} results:")
print("=" * 80)
for hit in filtered_results["hits"]["hits"]:
    print(f"Score: {hit['_score']:.4f}  |  Source: {hit['_source']['source']}  |  Chunk: {hit['_source']['chunk_id']}")
    print(f"  {hit['_source']['text'][:200]}...")
    print()

### Hybrid Search

**Hybrid search** combines vector similarity search with keyword-based (BM25) search. This is valuable because:
- **Vector search** captures semantic meaning — "automobile" matches "car"
- **Keyword search** captures exact terms — catches proper nouns, acronyms, and specific IDs that embeddings might miss

The `bool` query lets us combine a `knn` clause (vector) with a `match` clause (keyword) so both signals contribute to the final relevance score.

In [ ]:
# Hybrid search — combine vector similarity with keyword matching
query = "What is HNSW indexing in OpenSearch?"
query_embedding = get_embedding(query)

hybrid_results = oss_client.search(
    index=INDEX_NAME,
    body={
        "size": 3,
        "query": {
            "bool": {
                "should": [
                    {"knn": {"embedding": {"vector": query_embedding, "k": 3}}},
                    {"match": {"text": query}},
                ],
            }
        },
    },
)

print(f"Query: {query}")
print(f"Search type: Hybrid (vector + keyword)\n")
print(f"Top {len(hybrid_results['hits']['hits'])} results:")
print("=" * 80)
for hit in hybrid_results["hits"]["hits"]:
    print(f"Score: {hit['_score']:.4f}  |  Source: {hit['_source']['source']}  |  Chunk: {hit['_source']['chunk_id']}")
    print(f"  {hit['_source']['text'][:200]}...")
    print()

### Comparing Vector-Only vs Hybrid Search

Let's run the same query with both approaches to see how hybrid search can improve results by combining both signals.

In [ ]:
# Compare: vector-only vs hybrid for a query with a specific term
query = "What is the Converse API?"
query_embedding = get_embedding(query)

# Vector-only search
vector_results = oss_client.search(
    index=INDEX_NAME,
    body={
        "size": 3,
        "query": {"knn": {"embedding": {"vector": query_embedding, "k": 3}}},
    },
)

# Hybrid search
hybrid_results = oss_client.search(
    index=INDEX_NAME,
    body={
        "size": 3,
        "query": {
            "bool": {
                "should": [
                    {"knn": {"embedding": {"vector": query_embedding, "k": 3}}},
                    {"match": {"text": query}},
                ],
            }
        },
    },
)

print(f"Query: {query}\n")
print("VECTOR-ONLY results:")
print("-" * 60)
for hit in vector_results["hits"]["hits"]:
    print(f"  Score: {hit['_score']:.4f}  |  Source: {hit['_source']['source']}")

print("\nHYBRID results (vector + keyword):")
print("-" * 60)
for hit in hybrid_results["hits"]["hits"]:
    print(f"  Score: {hit['_score']:.4f}  |  Source: {hit['_source']['source']}")

## Cleanup

Delete the vector index we created. **Do not** delete the OpenSearch Serverless collection itself — it is shared infrastructure used by other labs and managed by `setup-resources.py`.

In [ ]:
# Delete the index (NOT the collection — that's shared infrastructure)
oss_client.indices.delete(index=INDEX_NAME)
print(f"Deleted index: {INDEX_NAME}")
print("OpenSearch Serverless collection 'genai-lab-vectors' is preserved for other labs.")

## Key Takeaways

1. **Embeddings capture semantic meaning** — texts about the same topic produce similar vectors even when they use different words, enabling search by meaning rather than keyword matching
2. **Chunking strategy directly impacts retrieval quality** — recursive chunking preserves document structure and produces more coherent chunks than naive fixed-size splitting
3. **Vector search enables natural language queries** — users ask questions in plain language and the system finds relevant documents without keyword engineering
4. **Metadata filtering narrows the search space** — filtering by source, date, or category improves both relevance and performance in production RAG pipelines
5. **Hybrid search combines the best of both worlds** — vector search captures semantic meaning while keyword search catches exact terms, proper nouns, and acronyms that embeddings may miss

## Key Concepts

| Concept | Definition |
|---------|-----------|
| Embedding | A dense vector representation of text that captures semantic meaning in a high-dimensional space; similar texts produce similar vectors |
| Vector dimension | The number of elements in an embedding vector (Titan V2 supports 256, 512, or 1024); higher dimensions capture more nuance but require more storage |
| Cosine similarity | A metric that measures the angle between two vectors; ranges from -1 to 1, where 1 means identical direction (same meaning) |
| Chunking | Splitting documents into smaller text segments before embedding; chunk size and strategy directly impact retrieval quality |
| knn_vector | An OpenSearch field type that stores vector embeddings and supports k-nearest neighbor similarity search |
| HNSW | Hierarchical Navigable Small World — a graph-based algorithm for fast approximate nearest neighbor search; used by OpenSearch with FAISS |
| Hybrid search | Combining vector similarity search (semantic) with keyword-based BM25 search (lexical) to improve retrieval quality and coverage |

## Exam Preparation — Common Exam Question Patterns

**Q: What is the purpose of text embeddings in a RAG pipeline?**
A: Text embeddings convert text into dense vector representations that capture semantic meaning. In a RAG pipeline, both documents and queries are embedded using the same model, enabling similarity search to find relevant documents based on meaning rather than exact keyword matches.

**Q: How does chunk size affect RAG retrieval quality?**
A: Chunks that are too large dilute the embedding with multiple topics, reducing search precision — a query about one topic may match a chunk that only partially covers it. Chunks that are too small lose context, reducing recall. The optimal size (typically 500-1000 characters) balances precision and context for your specific use case.

**Q: Why must the same embedding model be used for both documents and queries?**
A: Different embedding models produce vectors in different vector spaces with different dimensions and semantic mappings. A query embedded with Model A and documents embedded with Model B would produce meaningless similarity scores because the vectors are not comparable.

**Q: What is the advantage of hybrid search over pure vector search?**
A: Hybrid search combines vector similarity (semantic understanding) with keyword matching (exact term matching). Vector search captures meaning but may miss specific terms like product names or acronyms. Keyword search catches exact terms but misses semantic equivalents. Together they provide better coverage and relevance.

**Q: How does metadata filtering improve a RAG system?**
A: Metadata filtering restricts vector search to a subset of documents based on attributes like source, date, or access level. This improves relevance by excluding irrelevant documents from results, improves performance by reducing the search space, and enables access control by filtering based on user permissions.

## Cost Breakdown

| Resource | Usage | Est. Cost |
|----------|-------|-----------|
| Titan Embeddings V2 — Section A (sample texts + dimensions) | ~12 API calls, ~200 tokens | < $0.01 |
| Titan Embeddings V2 — Section C (chunk embeddings) | ~15 API calls, ~3K tokens | < $0.01 |
| Titan Embeddings V2 — Sections D-E (query embeddings) | ~6 API calls, ~200 tokens | < $0.01 |
| OpenSearch Serverless — OCU time | ~2 OCUs x 1 hour | ~$2-3 |
| **Total** | | **~$2-3** |

Titan Embeddings V2 pricing is $0.00002 per 1,000 input tokens — the embedding costs for this lab are negligible. The dominant cost is OpenSearch Serverless compute (OCU) time, which is billed per OCU-hour while the collection is active. The collection was created by `setup-resources.py` and will continue accruing charges until deleted via `cleanup-all.py`.